# 02 Preprocessing pipeline

Goal: turn the raw questionnaire data into model-ready data with **one reusable object**, and check that raw answers in -> personality type out works.

1. Load the cleaned data (`data/data_processed.csv`, written by `01_eda.ipynb`)
2. Stratified train/test split
3. Preprocessing pipeline (`ColumnTransformer` inside a `Pipeline`)
4. First models vs. a baseline

The cleaning runs once at the end of `01_eda.ipynb` (`personality/data.py`). The pipeline code lives in `personality/pipeline.py`, so notebooks and the app use exactly the same preprocessing.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from personality.config import PROCESSED_DATA_PATH, RANDOM_STATE, TEST_SIZE, CLASS_ORDER
from personality.data import split_features_target
from personality.pipeline import build_preprocessor, build_model_pipeline

pd.set_option("display.float_format", "{:.3f}".format)

## 1. Load the cleaned data

The cleaning decisions were applied once in `01_eda.ipynb`. Missing values are still in the data; the pipeline fills them.

In [2]:
df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"Rows after cleaning: {len(df)}  (1 all-zero row + 3 duplicates dropped in 01_eda.ipynb)")
print(f"Invalid ages set to NaN: {df['age'].isna().sum()}")

Rows after cleaning: 19715  (1 all-zero row + 3 duplicates dropped in 01_eda.ipynb)
Invalid ages set to NaN: 83


## 2. Stratified train/test split

`stratify=y` keeps the class shares (Moderate 43 %, ...) identical in train and test.
The split happens **before** any fitting, so the test set stays unseen (no data leakage).

In [3]:
X, y = split_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).reindex(CLASS_ORDER)

Train: (15772, 22)  Test: (3943, 22)


,train,test
target,,
Moderate,0.429,0.429
Resilient,0.313,0.312
Overcontroller,0.143,0.144
Undercontroller,0.115,0.115


## 3. The preprocessing pipeline

| Columns | Step 1 | Step 2 |
|---|---|---|
| 19 answers + `age` | fill gaps with median | standard scaling |
| `gender`, `hand` | fill gaps with most frequent | one-hot encoding |

The preprocessor is fitted on the **training data only**. Median, mean and categories are learned from train and then just applied to test.

In [4]:
preprocessor = build_preprocessor()
X_train_prepared = preprocessor.fit_transform(X_train)

print("Before:", X_train.shape, " After:", X_train_prepared.shape)
print("No gaps left:", not pd.DataFrame(X_train_prepared).isna().any().any())
list(preprocessor.get_feature_names_out())[-7:]  # gender/hand became 0/1 columns

Before: (15772, 22)  After: (15772, 26)
No gaps left: True


['num__age',
 'cat__gender_Female',
 'cat__gender_Male',
 'cat__gender_Other',
 'cat__hand_Both',
 'cat__hand_Left',
 'cat__hand_Right']

## 4. First models vs. baseline

The baseline always predicts the largest class (Moderate). Every real model has to beat it.
**Macro F1** averages the F1 score of all four classes equally, so small classes count as much as large ones.

In [5]:
models = {
    "Baseline (always Moderate)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
}

results = {}
fitted = {}
for name, classifier in models.items():
    model = build_model_pipeline(classifier)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    fitted[name] = model
    results[name] = {
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    }

pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)

,accuracy,macro_f1
Logistic Regression,0.850,0.758
Random Forest,0.845,0.757
Baseline (always Moderate),0.429,0.150


Per-class view for the Logistic Regression: which personality type is hardest?

In [6]:
print(classification_report(
    y_test, fitted["Logistic Regression"].predict(X_test), labels=CLASS_ORDER, digits=3
))

                 precision    recall  f1-score   support

       Moderate      0.799     0.910     0.851      1690
      Resilient      1.000     1.000     1.000      1232
 Overcontroller      0.789     0.795     0.792       566
Undercontroller      0.599     0.286     0.387       455

       accuracy                          0.850      3943
      macro avg      0.797     0.748     0.758      3943
   weighted avg      0.838     0.850     0.836      3943



## 5. Does the pipeline handle a new user with gaps?

A new app user with a missing `age` and `hand`: the pipeline fills the gaps and still predicts.

In [7]:
new_user = X_test.iloc[[0]].copy()
new_user[["age", "hand"]] = [None, None]
display(new_user)
fitted["Logistic Regression"].predict(new_user)

,N1,N2,N3,N4,N5,N6,N7,N8,N9,N10,...,E4,E5,E7,E9,E10,C4,A4,age,gender,hand
10494,4,2,5,2,3,2,2,2,3,4,...,2,5,5,5,2,2,5,None,Female,None


array(['Moderate'], dtype=object)

## Summary

- Cleaning removed 4 rows and set 83 invalid ages to `NaN`; the pipeline imputes them.
- The pipeline turns 22 raw columns into 26 model-ready columns (one-hot for `gender`/`hand`), with no gaps.
- Both real models clearly beat the baseline (accuracy 0.85 vs. 0.43, macro F1 0.76 vs. 0.15).
- **Resilient is predicted perfectly (F1 1.00):** its rule only depends on Neuroticism, and all 10 N questions are in the data.
- **Undercontroller is the weak spot (F1 0.39, recall 0.29):** its rule depends on Conscientiousness and Agreeableness, but the data has only one question each (C4, A4). The model misses 7 out of 10 Undercontrollers.
- This is why accuracy (0.85) looks better than macro F1 (0.76): accuracy hides the weak small class.
- Next step (Live Session 2): compare more models with cross-validation and tune the best ones.